# Notebook 03 — Training Analysis
**Sentinel v5.0 | Transparent training report with runner-up comparison + two improvement versions**

---

## What This Notebook Does

| Section | Content |
|---------|----------|
| §1 | Load feature parquet, apply train/test1/test2 splits |
| §2 | Baseline grid search (current system, direction=long) |
| §3 | Runner-up comparison table — top-3 candidates evaluated on test1/test2 |
| §4 | Baseline plots — P(win) distribution, equity curve, SHAP |
| §5 | **Version A** — Walk-forward CV + separate LONG/SHORT + ensemble + SHAP pruning |
| §6 | **Version B** — Version A + Platt probability calibration |
| §7 | Side-by-side comparison of Baseline vs Version A vs Version B |

---

### Why Two Versions?
- **Version A** applies the confirmed improvements: walk-forward validation, separate short models, ensemble, and feature pruning.
- **Version B** adds Platt scaling on top to test whether calibrated probabilities move the needle.
- Comparing both to the baseline quantifies the gain from each layer of improvement.

## §0 — Setup

In [ ]:
import os, sys, warnings, json
warnings.filterwarnings('ignore')

# ── Path resolution ────────────────────────────────────────────────────────────
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

import config
from utils.features   import build_all_features_incremental, _load_feature_parquet
from utils.labeling   import find_optimal_label_params, generate_labels
from utils.model_utils import (
    train_xgboost, evaluate_model, check_validity,
    get_shap_drivers, save_model, load_model, compute_profit_factor
)

# ── Coin to analyse ────────────────────────────────────────────────────────────
# Change TICKER to analyse a different altcoin.
TICKER = 'SOL/USDT'

print(f'Repo root : {REPO_ROOT}')
print(f'Ticker    : {TICKER}')
print(f'Config    : GRID_SEARCH_ESTIMATORS={config.GRID_SEARCH_ESTIMATORS}, FINAL_ESTIMATORS={config.FINAL_ESTIMATORS}')

## §1 — Data Preparation

In [ ]:
# ── Load feature parquet for the selected coin ────────────────────────────────
# If the parquet doesn't exist yet, run notebooks/01_data_and_features.ipynb first.

feat = _load_feature_parquet(TICKER)

if feat is None:
    raise FileNotFoundError(
        f'Feature parquet not found for {TICKER}.\n'
        f'Run notebook 01 (data & features) first, or call build_all_features_incremental() '
        f'from notebook 02 to generate it.'
    )

feat['timestamp'] = pd.to_datetime(feat['timestamp'], utc=True)
feat = feat.sort_values('timestamp').reset_index(drop=True)
print(f'Loaded {len(feat):,} rows | {feat["timestamp"].min()} → {feat["timestamp"].max()}')
feat.tail(3)

In [ ]:
# ── Train / Test1 / Test2 splits ───────────────────────────────────────────────
now = pd.Timestamp.utcnow()

train_end  = now - pd.Timedelta(days=config.TRAIN_END_DAYS)
test1_start = now - pd.Timedelta(days=config.TEST1[0])
test1_end   = now - pd.Timedelta(days=config.TEST1[1])
test2_start = now - pd.Timedelta(days=config.TEST2[0])
test2_end   = now - pd.Timedelta(days=config.TEST2[1])

# Anti-leakage: training data must end at least LABEL_HORIZON before test1
label_buffer = pd.Timedelta(hours=config.LABEL_HORIZON)

ts = feat['timestamp']
df_train = feat[ts <= (train_end - label_buffer)].copy()
df_test1 = feat[(ts > test1_start) & (ts <= test1_end)].copy()
df_test2 = feat[(ts > test2_start) & (ts <= test2_end)].copy()

print(f'Train : {len(df_train):>5,} rows | {df_train["timestamp"].min()} → {df_train["timestamp"].max()}')
print(f'Test1 : {len(df_test1):>5,} rows | {df_test1["timestamp"].min()} → {df_test1["timestamp"].max()}')
print(f'Test2 : {len(df_test2):>5,} rows | {df_test2["timestamp"].min()} → {df_test2["timestamp"].max()}')

# ── Feature columns ────────────────────────────────────────────────────────────
EXCLUDE = {'timestamp', 'open', 'high', 'low', 'close', 'volume', 'label'}
feature_cols = [c for c in feat.columns if c not in EXCLUDE and feat[c].dtype != object]
print(f'\nFeature columns: {len(feature_cols)}')

## §2 — Baseline Grid Search (current system, direction=long)

In [ ]:
%%time
# Baseline: current system grid search — direction=long only, single split
baseline_result = find_optimal_label_params(
    df_train, feature_cols, direction='long', verbose=True, n_jobs=4
)

print('\n── Baseline grid search result ──')
print(f'  Best params : tp={baseline_result["tp_pct"]:.3f} sl={baseline_result["sl_pct"]:.3f} '
      f'k1={baseline_result["k1"]} k2={baseline_result["k2"]}')
print(f'  Train PF    : {baseline_result["best_pf"]:.3f}  N={baseline_result["best_n"]}')
print(f'  Score       : {baseline_result["best_score"]:.3f}')

## §3 — Runner-Up Comparison Table

In [ ]:
def eval_candidate(params: dict, df_train, df_test1, df_test2, feature_cols, direction='long'):
    """Train a full (FINAL_ESTIMATORS) model for a candidate param set and evaluate on all splits."""
    tp, sl, k1, k2 = params['tp_pct'], params['sl_pct'], params['k1'], params['k2']
    threshold = config.LONG_THRESHOLD if direction == 'long' else config.SHORT_THRESHOLD

    def _labelled(df):
        labels = generate_labels(df, tp, sl, k1, k2, direction=direction)
        mask = labels.notna()
        X = df.loc[mask, feature_cols].copy()
        y = labels[mask].astype(int)
        return X, y

    X_tr, y_tr = _labelled(df_train)
    X_t1, y_t1 = _labelled(df_test1)
    X_t2, y_t2 = _labelled(df_test2)

    if len(y_tr) < 30:
        return None

    # Train with full estimators (production quality)
    model = train_xgboost(X_tr, y_tr)

    atr_t1 = X_t1['ATR_14_norm'].values if 'ATR_14_norm' in X_t1.columns else None
    atr_t2 = X_t2['ATR_14_norm'].values if 'ATR_14_norm' in X_t2.columns else None

    m_train = evaluate_model(model, X_tr, y_tr, threshold, direction, tp, sl, k1, k2,
                              atr_norm=X_tr['ATR_14_norm'].values if 'ATR_14_norm' in X_tr.columns else None)
    m_test1 = evaluate_model(model, X_t1, y_t1, threshold, direction, tp, sl, k1, k2, atr_norm=atr_t1)
    m_test2 = evaluate_model(model, X_t2, y_t2, threshold, direction, tp, sl, k1, k2, atr_norm=atr_t2)

    valid = check_validity(m_test1, m_test2)

    return {
        'params':  params,
        'model':   model,
        'X_tr': X_tr, 'y_tr': y_tr,
        'X_t1': X_t1, 'y_t1': y_t1,
        'X_t2': X_t2, 'y_t2': y_t2,
        'train': m_train,
        'test1': m_test1,
        'test2': m_test2,
        'valid': valid,
    }


# ── Evaluate all runner-ups (top-3 from grid search) ──────────────────────────
runner_up_evals = []
for rank, ru in enumerate(baseline_result['runner_ups']):
    print(f'Evaluating rank {rank+1}: tp={ru["params"]["tp_pct"]:.3f} '
          f'sl={ru["params"]["sl_pct"]:.3f} k1={ru["params"]["k1"]} k2={ru["params"]["k2"]} ...')
    ev = eval_candidate(ru['params'], df_train, df_test1, df_test2, feature_cols, direction='long')
    if ev is not None:
        ev['rank'] = rank + 1
        ev['grid_score'] = ru['score']
        ev['grid_pf']    = ru['pf']
        runner_up_evals.append(ev)

print('\nDone.')

In [ ]:
# ── Display runner-up comparison table ────────────────────────────────────────
rows = []
for ev in runner_up_evals:
    p = ev['params']
    rows.append({
        'Rank':        ev['rank'],
        'tp_pct':      f"{p['tp_pct']:.3f}",
        'sl_pct':      f"{p['sl_pct']:.3f}",
        'k1':          p['k1'],
        'k2':          p['k2'],
        'GridScore':   ev['grid_score'],
        'Train_PF':    ev['train']['PF'],
        'Train_N':     ev['train']['trade_count'],
        'Train_WR':    f"{ev['train']['win_rate']:.1%}",
        'Test1_PF':    ev['test1']['PF'],
        'Test1_N':     ev['test1']['trade_count'],
        'Test2_PF':    ev['test2']['PF'],
        'Test2_N':     ev['test2']['trade_count'],
        'Stability':   f"{ev['test2']['PF'] / ev['test1']['PF']:.2f}" if ev['test1']['PF'] > 0 else 'N/A',
        'Valid':       '✅' if ev['valid'] else '❌',
    })

df_runnerup = pd.DataFrame(rows)
print(f'Runner-up comparison table — {TICKER} (LONG direction)')
print('Gates: Test1 PF≥1.20, Test2/Test1≥0.70, Test1 N≥20')
print('=' * 80)
display(df_runnerup.set_index('Rank'))

## §4 — Baseline Plots

In [ ]:
# Use the #1 ranked candidate for plots
best_ev = runner_up_evals[0] if runner_up_evals else None

if best_ev is None:
    print('No valid candidate found — check data coverage.')
else:
    model   = best_ev['model']
    X_t1    = best_ev['X_t1']
    y_t1    = best_ev['y_t1']
    p       = best_ev['params']
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{TICKER} | Baseline Model (rank 1) | tp={p["tp_pct"]:.3f} sl={p["sl_pct"]:.3f}',
                 fontsize=13)

    # ── Plot 1: P(win) distribution ────────────────────────────────────────────
    ax = axes[0]
    p_win = model.predict_proba(X_t1)[:, 1]
    ax.hist(p_win[y_t1 == 1], bins=20, alpha=0.6, color='green', label='Actual Win')
    ax.hist(p_win[y_t1 == 0], bins=20, alpha=0.6, color='red',   label='Actual Loss')
    ax.axvline(config.LONG_THRESHOLD,  color='green', ls='--', lw=1.5, label=f'LONG threshold={config.LONG_THRESHOLD}')
    ax.axvline(config.SHORT_THRESHOLD, color='red',   ls='--', lw=1.5, label=f'SHORT threshold={config.SHORT_THRESHOLD}')
    ax.set_xlabel('P(win)')
    ax.set_ylabel('Count')
    ax.set_title('P(win) Distribution — Test1')
    ax.legend(fontsize=8)

    # ── Plot 2: Equity curve simulation (Test1) ────────────────────────────────
    ax = axes[1]
    mask_long = p_win >= config.LONG_THRESHOLD
    atr_norm = X_t1['ATR_14_norm'].values if 'ATR_14_norm' in X_t1.columns else None
    if atr_norm is not None:
        profits = np.where(y_t1.values == 1,
                           p['tp_pct'] + p['k1'] * atr_norm,
                           -(p['sl_pct'] + p['k2'] * atr_norm))
    else:
        profits = np.where(y_t1.values == 1, p['tp_pct'], -p['sl_pct'])
    triggered_profits = profits[mask_long]
    cumulative = np.cumsum(triggered_profits)
    ax.plot(cumulative, color='steelblue', lw=1.5)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.fill_between(range(len(cumulative)), cumulative, 0,
                    where=cumulative >= 0, alpha=0.2, color='green')
    ax.fill_between(range(len(cumulative)), cumulative, 0,
                    where=cumulative < 0, alpha=0.2, color='red')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative Return (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title(f'Equity Curve — Test1 ({mask_long.sum()} LONG trades)')

    # ── Plot 3: SHAP feature importance ───────────────────────────────────────
    ax = axes[2]
    try:
        import shap
        explainer  = shap.TreeExplainer(model)
        shap_vals  = explainer.shap_values(X_t1)
        mean_abs   = np.abs(shap_vals).mean(axis=0)
        top15_idx  = np.argsort(mean_abs)[::-1][:15]
        top15_names = [X_t1.columns[i] for i in top15_idx]
        top15_vals  = mean_abs[top15_idx]
        ax.barh(top15_names[::-1], top15_vals[::-1], color='steelblue')
        ax.set_xlabel('Mean |SHAP value|')
        ax.set_title('Feature Importance (SHAP) — Test1')
    except Exception as e:
        ax.text(0.5, 0.5, f'SHAP unavailable:\n{e}', ha='center', va='center',
                transform=ax.transAxes)

    plt.tight_layout()
    plt.savefig(os.path.join(REPO_ROOT, 'notebooks', f'03_baseline_{TICKER.replace("/","_")}.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Test1  — PF={best_ev["test1"]["PF"]:.3f}  N={best_ev["test1"]["trade_count"]}  WR={best_ev["test1"]["win_rate"]:.1%}')
    print(f'Test2  — PF={best_ev["test2"]["PF"]:.3f}  N={best_ev["test2"]["trade_count"]}  WR={best_ev["test2"]["win_rate"]:.1%}')
    print(f'Valid  — {"PASS" if best_ev["valid"] else "FAIL"}')

---
## §5 — Version A: Walk-forward CV + Separate LONG/SHORT + Ensemble + SHAP Pruning

**Improvements applied:**
- **I1 — Walk-forward cross-validation**: 5 rolling windows instead of a single train/test split
- **I2 — Separate LONG & SHORT models**: Each direction gets its own grid search, labels, and trained XGBoost. P(win)≤0.40 on the LONG model is meaningless for shorts — this fixes it.
- **I3 — Ensemble of top-3 runner-ups**: Average `predict_proba` across the top-3 label param combos. Disagreement between models = low-confidence signal.
- **I4 — SHAP feature pruning**: Drop near-zero SHAP features before retraining the final model.

In [ ]:
# ── Helper: walk-forward windows ──────────────────────────────────────────────
def make_walk_forward_windows(feat_df, n_folds=5, test_days=30, gap_days=2):
    """
    Build n_folds walk-forward windows.
    Each fold: train = all data before fold's test start - gap, test = `test_days` days.
    Returns list of (df_train, df_test) tuples.
    """
    feat_df = feat_df.copy()
    feat_df['timestamp'] = pd.to_datetime(feat_df['timestamp'], utc=True)
    max_ts = feat_df['timestamp'].max()

    windows = []
    for fold in range(n_folds):
        # Test window: walk backwards from max_ts
        test_end   = max_ts - pd.Timedelta(days=fold * test_days)
        test_start = test_end - pd.Timedelta(days=test_days)
        train_end  = test_start - pd.Timedelta(days=gap_days)  # gap prevents leakage

        df_tr = feat_df[feat_df['timestamp'] <= train_end].copy()
        df_te = feat_df[(feat_df['timestamp'] > test_start) &
                        (feat_df['timestamp'] <= test_end)].copy()

        if len(df_tr) > 200 and len(df_te) > 20:
            windows.append((df_tr, df_te))

    return list(reversed(windows))  # chronological order


# ── Helper: SHAP feature pruning ──────────────────────────────────────────────
def prune_features_by_shap(model, X, feature_cols, threshold_pct=0.005):
    """
    Remove features whose mean |SHAP value| is below `threshold_pct` of the max.
    Returns pruned feature list.
    """
    try:
        import shap
        explainer = shap.TreeExplainer(model)
        # Use a sample for speed (max 500 rows)
        X_sample = X.sample(min(500, len(X)), random_state=42)
        shap_vals = explainer.shap_values(X_sample)
        mean_abs  = np.abs(shap_vals).mean(axis=0)
        threshold = mean_abs.max() * threshold_pct
        keep_mask = mean_abs >= threshold
        pruned = [c for c, keep in zip(feature_cols, keep_mask) if keep]
        print(f'  SHAP pruning: {len(feature_cols)} → {len(pruned)} features '
              f'(dropped {len(feature_cols)-len(pruned)} near-zero)')
        return pruned
    except Exception as e:
        print(f'  SHAP pruning failed: {e} — keeping all features')
        return feature_cols


# ── Helper: train ensemble (top-3 runner-ups) ─────────────────────────────────
def train_ensemble(df_train, feature_cols, direction):
    """
    Run grid search, take top-3 runner-up param combos, train one model per combo.
    Returns (models_list, pruned_feature_cols, grid_result).
    Step 1: Grid search with GRID_SEARCH_ESTIMATORS (fast ranking).
    Step 2: For each top-3: SHAP-prune features, retrain with FINAL_ESTIMATORS.
    """
    result = find_optimal_label_params(
        df_train, feature_cols, direction=direction, verbose=False, n_jobs=4
    )

    if not result['runner_ups']:
        return [], feature_cols, result

    models = []
    final_feat_cols = feature_cols  # will be set by first successful prune

    for i, ru in enumerate(result['runner_ups']):
        p = ru['params']
        labels = generate_labels(df_train, p['tp_pct'], p['sl_pct'], p['k1'], p['k2'],
                                  direction=direction)
        mask = labels.notna()
        X = df_train.loc[mask, feature_cols].copy()
        y = labels[mask].astype(int)

        if len(y) < 30:
            continue

        # I4: SHAP pruning on initial model
        init_model = train_xgboost(X, y, n_estimators=50)
        pruned_cols = prune_features_by_shap(init_model, X, list(X.columns))
        if i == 0:
            final_feat_cols = pruned_cols  # use first (best) prune for consistency

        # I1 final model: full estimators on pruned features
        X_pruned = X[pruned_cols] if pruned_cols else X
        final_model = train_xgboost(X_pruned, y)
        models.append({
            'model':       final_model,
            'params':      p,
            'feature_cols': pruned_cols or feature_cols,
        })

    return models, final_feat_cols, result


def ensemble_predict_proba(models_list, X_full, fallback_feature_cols):
    """
    Average predict_proba across all models in the ensemble.
    Each model may have different pruned feature sets — subset X accordingly.
    """
    probs = []
    for m in models_list:
        fc = m['feature_cols']
        available = [c for c in fc if c in X_full.columns]
        if not available:
            continue
        p = m['model'].predict_proba(X_full[available])[:, 1]
        probs.append(p)
    if not probs:
        return np.full(len(X_full), 0.5)
    return np.mean(probs, axis=0)


print('Version A helpers defined.')

In [ ]:
%%time
# ── Version A: Walk-forward evaluation ────────────────────────────────────────
N_FOLDS    = 5
TEST_DAYS  = 30

wf_windows = make_walk_forward_windows(feat, n_folds=N_FOLDS, test_days=TEST_DAYS)
print(f'Walk-forward windows: {len(wf_windows)} folds x ~{TEST_DAYS} days each\n')

vA_long_results  = []
vA_short_results = []

for fold_idx, (df_tr, df_te) in enumerate(wf_windows):
    print(f'─ Fold {fold_idx+1}/{len(wf_windows)} │ train={len(df_tr):,} test={len(df_te):,} rows')

    # I2: Separate LONG model
    print('  Training LONG  ensemble...')
    long_models, long_feat_cols, long_gs = train_ensemble(df_tr, feature_cols, 'long')

    # I2: Separate SHORT model (its own grid search + labels)
    print('  Training SHORT ensemble...')
    short_models, short_feat_cols, short_gs = train_ensemble(df_tr, feature_cols, 'short')

    # Evaluate LONG on test fold
    if long_models:
        # Use best params from LONG grid search for test labels
        best_lp = long_gs
        labels_te = generate_labels(df_te, best_lp['tp_pct'], best_lp['sl_pct'],
                                     best_lp['k1'], best_lp['k2'], direction='long')
        mask_te  = labels_te.notna()
        X_te     = df_te.loc[mask_te, feature_cols].copy()
        y_te     = labels_te[mask_te].astype(int)

        if len(y_te) >= 10:
            p_win_ens = ensemble_predict_proba(long_models, X_te, long_feat_cols)
            triggered = p_win_ens >= config.LONG_THRESHOLD
            if triggered.sum() > 0:
                atr_n = X_te['ATR_14_norm'].values if 'ATR_14_norm' in X_te.columns else None
                pf_trades = []
                for j, trig in enumerate(triggered):
                    if not trig:
                        continue
                    atr_j = atr_n[j] if atr_n is not None else 0
                    gain = (best_lp['tp_pct'] + best_lp['k1'] * atr_j) if y_te.iloc[j] == 1 \
                           else -(best_lp['sl_pct'] + best_lp['k2'] * atr_j)
                    pf_trades.append((y_te.iloc[j], atr_j))

                gross_p = sum((best_lp['tp_pct'] + best_lp['k1'] * (atr_n[j] if atr_n is not None else 0))
                              for j, t in enumerate(triggered) if t and y_te.iloc[j] == 1)
                gross_l = sum((best_lp['sl_pct'] + best_lp['k2'] * (atr_n[j] if atr_n is not None else 0))
                              for j, t in enumerate(triggered) if t and y_te.iloc[j] == 0)
                pf = gross_p / gross_l if gross_l > 0 else float('inf')
                n_tr = int(triggered.sum())
                wr   = float(y_te.values[triggered].mean())
                vA_long_results.append({'fold': fold_idx+1, 'PF': pf, 'N': n_tr, 'WR': wr})
                print(f'    LONG  → PF={pf:.3f}  N={n_tr}  WR={wr:.1%}')

    # Evaluate SHORT on test fold
    if short_models:
        best_sp = short_gs
        labels_te_s = generate_labels(df_te, best_sp['tp_pct'], best_sp['sl_pct'],
                                       best_sp['k1'], best_sp['k2'], direction='short')
        mask_te_s = labels_te_s.notna()
        X_te_s    = df_te.loc[mask_te_s, feature_cols].copy()
        y_te_s    = labels_te_s[mask_te_s].astype(int)

        if len(y_te_s) >= 10:
            p_win_s = ensemble_predict_proba(short_models, X_te_s, short_feat_cols)
            triggered_s = p_win_s <= config.SHORT_THRESHOLD
            if triggered_s.sum() > 0:
                atr_ns = X_te_s['ATR_14_norm'].values if 'ATR_14_norm' in X_te_s.columns else None
                gross_p_s = sum((best_sp['tp_pct'] + best_sp['k1'] * (atr_ns[j] if atr_ns is not None else 0))
                                for j, t in enumerate(triggered_s) if t and y_te_s.iloc[j] == 1)
                gross_l_s = sum((best_sp['sl_pct'] + best_sp['k2'] * (atr_ns[j] if atr_ns is not None else 0))
                                for j, t in enumerate(triggered_s) if t and y_te_s.iloc[j] == 0)
                pf_s = gross_p_s / gross_l_s if gross_l_s > 0 else float('inf')
                n_s  = int(triggered_s.sum())
                wr_s = float(y_te_s.values[triggered_s].mean())
                vA_short_results.append({'fold': fold_idx+1, 'PF': pf_s, 'N': n_s, 'WR': wr_s})
                print(f'    SHORT → PF={pf_s:.3f}  N={n_s}  WR={wr_s:.1%}')

print('\nVersion A complete.')

In [ ]:
# ── Version A summary ─────────────────────────────────────────────────────────
def summarise_wf(results, label):
    if not results:
        print(f'{label}: No results.')
        return
    df = pd.DataFrame(results)
    print(f'\n── {label} Walk-Forward Summary ──')
    print(df.to_string(index=False))
    print(f'  Mean PF : {df["PF"].mean():.3f}')
    print(f'  Std  PF : {df["PF"].std():.3f}')
    print(f'  Mean N  : {df["N"].mean():.1f}')
    print(f'  Mean WR : {df["WR"].mean():.1%}')
    folds_pass = (df['PF'] >= config.MIN_PF_TEST1).sum()
    print(f'  Folds passing PF≥{config.MIN_PF_TEST1}: {folds_pass}/{len(df)}')
    return df

vA_long_df  = summarise_wf(vA_long_results,  'Version A — LONG')
vA_short_df = summarise_wf(vA_short_results, 'Version A — SHORT')

---
## §6 — Version B: Version A + Platt Probability Calibration (I5)

**Additional improvement over Version A:**
- **I5 — Platt scaling**: Wraps the XGBoost model with `CalibratedClassifierCV(method='sigmoid')` on a held-out calibration split. This corrects the raw `predict_proba` output so that a predicted P(win)=0.60 actually corresponds to ~60% real win rate.
- **Why this matters**: An uncalibrated model might produce P(win)=0.65 for trades that only win 52% of the time. Calibrated probabilities make the 0.60/0.40 thresholds semantically correct.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

def train_calibrated_model(X_train, y_train, feature_cols, direction, n_calib_folds=3):
    """
    Train a Platt-scaled calibrated XGBoost model.
    Uses CalibratedClassifierCV with cross-validation (avoids needing a separate split).
    """
    from xgboost import XGBClassifier
    base = XGBClassifier(
        n_estimators=config.FINAL_ESTIMATORS,
        max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, eval_metric='logloss',
        use_label_encoder=False, verbosity=0, random_state=42,
    )
    calibrated = CalibratedClassifierCV(base, method='sigmoid', cv=n_calib_folds)
    calibrated.fit(X_train, y_train)
    return calibrated


print('Version B helpers defined.')

In [ ]:
%%time
# ── Version B: Walk-forward evaluation with calibrated models ─────────────────
vB_long_results  = []
vB_short_results = []

for fold_idx, (df_tr, df_te) in enumerate(wf_windows):
    print(f'─ Fold {fold_idx+1}/{len(wf_windows)}')

    for direction, results_list, threshold in [
        ('long',  vB_long_results,  config.LONG_THRESHOLD),
        ('short', vB_short_results, config.SHORT_THRESHOLD),
    ]:
        gs = find_optimal_label_params(
            df_tr, feature_cols, direction=direction, verbose=False, n_jobs=4
        )
        if not gs['runner_ups']:
            continue

        best_p = gs  # use best params
        labels_tr = generate_labels(df_tr, best_p['tp_pct'], best_p['sl_pct'],
                                     best_p['k1'], best_p['k2'], direction=direction)
        mask_tr = labels_tr.notna()
        X_tr_d  = df_tr.loc[mask_tr, feature_cols].copy()
        y_tr_d  = labels_tr[mask_tr].astype(int)

        if len(y_tr_d) < 30 or y_tr_d.sum() < 5:
            continue

        # I4: SHAP pruning
        init_m = train_xgboost(X_tr_d, y_tr_d, n_estimators=50)
        pruned_cols = prune_features_by_shap(init_m, X_tr_d, list(X_tr_d.columns))
        X_tr_pruned = X_tr_d[pruned_cols] if pruned_cols else X_tr_d

        # I5: Platt calibration (replaces plain train_xgboost)
        calib_model = train_calibrated_model(X_tr_pruned, y_tr_d, pruned_cols, direction)

        # Evaluate on test fold
        labels_te = generate_labels(df_te, best_p['tp_pct'], best_p['sl_pct'],
                                     best_p['k1'], best_p['k2'], direction=direction)
        mask_te = labels_te.notna()
        X_te_d  = df_te.loc[mask_te, feature_cols].copy()
        y_te_d  = labels_te[mask_te].astype(int)

        if len(y_te_d) < 10:
            continue

        X_te_pruned = X_te_d[[c for c in pruned_cols if c in X_te_d.columns]]
        p_win_cal   = calib_model.predict_proba(X_te_pruned)[:, 1]

        if direction == 'long':
            triggered = p_win_cal >= threshold
        else:
            triggered = p_win_cal <= threshold

        if triggered.sum() == 0:
            continue

        atr_n_te = X_te_d['ATR_14_norm'].values if 'ATR_14_norm' in X_te_d.columns else None
        gross_p = sum(
            (best_p['tp_pct'] + best_p['k1'] * (atr_n_te[j] if atr_n_te is not None else 0))
            for j, t in enumerate(triggered) if t and y_te_d.iloc[j] == 1
        )
        gross_l = sum(
            (best_p['sl_pct'] + best_p['k2'] * (atr_n_te[j] if atr_n_te is not None else 0))
            for j, t in enumerate(triggered) if t and y_te_d.iloc[j] == 0
        )
        pf   = gross_p / gross_l if gross_l > 0 else float('inf')
        n_t  = int(triggered.sum())
        wr_t = float(y_te_d.values[triggered].mean())
        results_list.append({'fold': fold_idx+1, 'PF': pf, 'N': n_t, 'WR': wr_t})
        print(f'  {direction.upper():5s} → PF={pf:.3f}  N={n_t}  WR={wr_t:.1%}')

print('\nVersion B complete.')

In [ ]:
# ── Version B summary ─────────────────────────────────────────────────────────
vB_long_df  = summarise_wf(vB_long_results,  'Version B — LONG  (calibrated)')
vB_short_df = summarise_wf(vB_short_results, 'Version B — SHORT (calibrated)')

---
## §7 — Side-by-Side Comparison: Baseline vs Version A vs Version B

In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────────
def mean_or_na(df, col):
    if df is None or len(df) == 0:
        return 'N/A'
    return f"{df[col].mean():.3f}"

def folds_pass(df):
    if df is None or len(df) == 0:
        return 'N/A'
    return f"{(df['PF'] >= config.MIN_PF_TEST1).sum()}/{len(df)}"

# Baseline uses single split (test1)
baseline_pf_t1  = best_ev['test1']['PF']  if best_ev else float('nan')
baseline_pf_t2  = best_ev['test2']['PF']  if best_ev else float('nan')
baseline_n_t1   = best_ev['test1']['trade_count'] if best_ev else 0
baseline_valid  = '✅' if (best_ev and best_ev['valid']) else '❌'

rows = [
    {
        'Version':          'Baseline (current)',
        'Direction':        'LONG only',
        'Validation':       'Single split',
        'Ensemble':         'No',
        'SHAP Prune':       'No',
        'Calibrated':       'No',
        'Test1 PF':         f'{baseline_pf_t1:.3f}',
        'Test2 PF':         f'{baseline_pf_t2:.3f}',
        'Test1 N':          baseline_n_t1,
        'Stability':        f'{baseline_pf_t2/baseline_pf_t1:.2f}' if baseline_pf_t1 > 0 else 'N/A',
        'Valid?':           baseline_valid,
    },
    {
        'Version':          'Version A (I1+I2+I3+I4)',
        'Direction':        'LONG + SHORT',
        'Validation':       f'{N_FOLDS}-fold walk-forward',
        'Ensemble':         'Top-3 runner-ups',
        'SHAP Prune':       'Yes',
        'Calibrated':       'No',
        'Test1 PF':         f"{mean_or_na(vA_long_df, 'PF')} (LONG)",
        'Test2 PF':         f"{mean_or_na(vA_short_df, 'PF')} (SHORT)",
        'Test1 N':          f"{mean_or_na(vA_long_df, 'N')} / {mean_or_na(vA_short_df, 'N')}",
        'Stability':        f'{folds_pass(vA_long_df)} folds pass (LONG)',
        'Valid?':           '—',
    },
    {
        'Version':          'Version B (A + I5 calibration)',
        'Direction':        'LONG + SHORT',
        'Validation':       f'{N_FOLDS}-fold walk-forward',
        'Ensemble':         'Single best model',
        'SHAP Prune':       'Yes',
        'Calibrated':       'Platt sigmoid',
        'Test1 PF':         f"{mean_or_na(vB_long_df, 'PF')} (LONG)",
        'Test2 PF':         f"{mean_or_na(vB_short_df, 'PF')} (SHORT)",
        'Test1 N':          f"{mean_or_na(vB_long_df, 'N')} / {mean_or_na(vB_short_df, 'N')}",
        'Stability':        f'{folds_pass(vB_long_df)} folds pass (LONG)',
        'Valid?':           '—',
    },
]

df_comparison = pd.DataFrame(rows).set_index('Version')
print(f'\n══ Comparison: {TICKER} ══')
display(df_comparison)

In [ ]:
# ── Visual comparison plot ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{TICKER} — Baseline vs Version A vs Version B', fontsize=13)

# Plot 1: Mean PF per version (LONG only for fair baseline comparison)
ax = axes[0]
versions_long = ['Baseline\n(test1)'] + \
    [f'Vrs A\nFold {r["fold"]}' for r in vA_long_results] + \
    [f'Vrs B\nFold {r["fold"]}' for r in vB_long_results]
pfs_long = [baseline_pf_t1] + \
    [r['PF'] for r in vA_long_results] + \
    [r['PF'] for r in vB_long_results]
colors = ['grey'] + ['steelblue'] * len(vA_long_results) + ['darkorange'] * len(vB_long_results)
ax.bar(versions_long, pfs_long, color=colors, alpha=0.8)
ax.axhline(config.MIN_PF_TEST1, color='red', ls='--', lw=1.5, label=f'Min PF={config.MIN_PF_TEST1}')
ax.axhline(1.0, color='black', ls='-', lw=0.8)
ax.set_ylabel('Profit Factor')
ax.set_title('LONG — PF per fold (grey=Baseline, blue=VrsA, orange=VrsB)')
ax.legend()
ax.tick_params(axis='x', labelsize=7)

# Plot 2: SHORT signal coverage (Version A vs B only — baseline has none)
ax = axes[1]
if vA_short_results or vB_short_results:
    vA_s = [r['PF'] for r in vA_short_results]
    vB_s = [r['PF'] for r in vB_short_results]
    x = np.arange(max(len(vA_s), len(vB_s)))
    if vA_s:
        ax.bar(x[:len(vA_s)] - 0.2, vA_s, width=0.35, label='Version A', color='steelblue', alpha=0.8)
    if vB_s:
        ax.bar(x[:len(vB_s)] + 0.2, vB_s, width=0.35, label='Version B', color='darkorange', alpha=0.8)
    ax.axhline(config.MIN_PF_TEST1, color='red', ls='--', lw=1.5)
    ax.axhline(1.0, color='black', ls='-', lw=0.8)
    ax.set_xlabel('Fold')
    ax.set_ylabel('Profit Factor')
    ax.set_title('SHORT — PF per fold (new in v.A/B, not in baseline)')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No SHORT signals fired\nin any fold.', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
    ax.set_title('SHORT — No signals')

plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'notebooks', f'03_comparison_{TICKER.replace("/","_")}.png'),
            dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Interpretation ────────────────────────────────────────────────────────────
print('═' * 70)
print('INTERPRETATION')
print('═' * 70)

def interpret_pf(label, pf_str):
    try:
        pf = float(pf_str.split()[0])
        if pf >= 1.20:
            return f'{label}: PF={pf:.3f} ✅  Passes validity gate'
        elif pf >= 1.0:
            return f'{label}: PF={pf:.3f} ⚠️  Profitable but below PF≥1.20 gate'
        else:
            return f'{label}: PF={pf:.3f} ❌  Unprofitable'
    except Exception:
        return f'{label}: {pf_str}'

print(interpret_pf('Baseline Test1', f'{baseline_pf_t1:.3f}'))
print(interpret_pf('Baseline Test2', f'{baseline_pf_t2:.3f}'))

if vA_long_df is not None and len(vA_long_df):
    mean_a = vA_long_df['PF'].mean()
    pass_a = (vA_long_df['PF'] >= config.MIN_PF_TEST1).sum()
    print(f'Version A LONG mean PF={mean_a:.3f}  ({pass_a}/{len(vA_long_df)} folds pass)')
if vA_short_df is not None and len(vA_short_df):
    mean_as = vA_short_df['PF'].mean()
    print(f'Version A SHORT mean PF={mean_as:.3f}  (new coverage not in baseline)')
if vB_long_df is not None and len(vB_long_df):
    mean_b = vB_long_df['PF'].mean()
    pass_b = (vB_long_df['PF'] >= config.MIN_PF_TEST1).sum()
    print(f'Version B LONG mean PF={mean_b:.3f}  ({pass_b}/{len(vB_long_df)} folds pass)')

print('\nKey questions to answer from results above:')
print('  1. Does walk-forward PF distribution show regime stability or high variance?')
print('  2. Does the SHORT model produce valid PF > 1.20 in at least 3/5 folds?')
print('  3. Does Platt calibration (V.B) improve or hurt PF vs V.A?')
print('  4. How many features were dropped by SHAP pruning? Did it hurt signal quality?')
print('═' * 70)